# 1차 실험 — 16개 조합 + 추가 시도 (형태소 질의 보강 · 청크 머리말 달기)

원문 → et5 교정 → 원문 vs 교정문 diff → 질의 만들기 → 검색 → 근거 조항 최대 3개 vs 정답 조항 채점

**1차-1. 16개 조합** (셀 0~14) — 4개 축 × 2가지 방식

| 축 | 베이스라인 방식 | 비교한 방식 |
|---|---|---|
| 쿼리 | kanana 2.1B 쿼리 재작성 | 재작성 없음 (diff 그대로) |
| 임베딩 | bge-m3 | KURE-v1 |
| 검색 | 벡터 서치만 | 하이브리드 검색 (벡터 50 + BM25 50 → RRF 20 → bge-reranker-v2-m3 리랭킹) |
| 청킹 | 고정 길이 500으로 청킹 | 구조+semantic 청킹 |

**1차-2. 추가 시도** (셀 15) — 1차 최고 4번 조합에 형태소 질의 보강 · 청크 머리말 달기를 더해 비교

**지표**: 제2항 규칙이 없으므로 검색 결과만 채점한다 — R@3, R@3 (제2항 제외), R@1

**실행 환경**: Google Colab (GPU T4). 셀을 위에서부터 차례로 실행한다. 무거운 결과(et5 교정, 재작성, 검색 순위)는 Drive `cache/`에 저장되어 다시 실행할 때 건너뛴다.

**평가 데이터**: AI허브 맞춤법 오류 데이터에 근거 조항을 직접 라벨로 단 엑셀(`rag_eval/맞춤법_RAG_평가셋_검수표_1000_라벨.xlsx`)이 필요하다. AI허브 이용 약관상 재배포가 금지돼 저장소에는 포함하지 않았다. 1·2·3차 모두 같은 500문항(dev 146 / test 354)을 쓰고, et5가 정답대로 고친 test 250문항으로 채점한다.

결과 해석은 [README](../README.md)에 정리했다.

In [ ]:
# =====================================================================
# [셀 0] 설치 — 실행 후 '런타임 → 세션 다시 시작'
#   requirements.txt 와 같은 버전 범위 (세 노트북 공통)
#   transformers 5.x는 kanana 설정 검증에서 오류가 나므로 4.x로 고정
# =====================================================================
!pip -q install "transformers>=4.51,<5" "sentence-transformers>=3.0,<6" "kiwipiepy>=0.18" "rank_bm25>=0.2.2" "openpyxl>=3.1" "jinja2>=3.1" "beautifulsoup4>=4.12"

In [ ]:
# =====================================================================
# [셀 1] 설정 — 경로·모델·파라미터는 여기서만 바꾼다
# =====================================================================
import os, re, json, time, hashlib, unicodedata, difflib, gc, glob
from collections import defaultdict, Counter
from math import comb
import numpy as np
import pandas as pd
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/rag_eval"
except ImportError:                                   # 로컬 실행
    BASE = os.path.abspath("./rag_eval")

EXCEL_PATH = f"{BASE}/맞춤법_RAG_평가셋_검수표_1000_라벨.xlsx"   # 라벨 엑셀(저장소에 포함하지 않음)
RULES_TXT  = f"{BASE}/corpus/한글맞춤법_본문.txt"                 # 맞춤법 본문(없으면 셀 7이 받아서 저장)
RULES_URL  = "https://korean.go.kr/kornorms/regltn/regltnView.do?regltn_code=0001"
OUT_DIR, CACHE_DIR = f"{BASE}/results", f"{BASE}/cache"
RANK_DIR = f"{CACHE_DIR}/ranks"
for d in [OUT_DIR, CACHE_DIR, RANK_DIR, os.path.dirname(RULES_TXT)]:
    os.makedirs(d, exist_ok=True)

# 평가 문항: '사용' 라벨 문항 중 무작위 N개(추출 그룹 × dev/test 비율 유지)
N_SAMPLE, SAMPLE_SEED = 500, 42

# 교정 모델(et5)
ET5_MODEL  = "j5ng/et5-typos-corrector"
ET5_PREFIX = "맞춤법을 고쳐주세요: "
ET5_BEAMS, ET5_MAX_LEN, ET5_BATCH = 5, 256, 16

# diff: 바뀐 어절 앞뒤로 붙일 어절 수
CTX = 1

# 임베딩·리랭커
EMBED_MODELS = {"bge-m3": "BAAI/bge-m3", "KURE": "nlpai-lab/KURE-v1"}
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
SEM_SPLIT_MODEL = "BAAI/bge-m3"      # 구조+semantic 청킹의 의미 분할용(임베딩 축과 무관하게 고정)

# 검색
VEC_TOPN, BM25_TOPN, RRF_K, RRF_TOPN = 50, 50, 60, 20
TOPK_EVAL = 3

# 청킹
FIXED_LEN = 500
SEM_PCT, SEM_MIN, SEM_MAX = 10, 200, 800     # 하위 10% 유사도 지점에서 자르고, 200~800자 유지

# 쿼리 재작성(kanana 2.1B) — 조항 번호를 스스로 판단하지 않게, 명사형으로
REWRITE_MODEL = "kakaocorp/kanana-1.5-2.1b-instruct-2505"
REWRITE_SYSTEM = """당신은 맞춤법 규정 문서를 검색하기 위한 검색어를 만드는 도구입니다.
입력은 '교정 전 → 교정 후' 문장 조각입니다. 아래 규칙을 반드시 지키세요.
1. 조항 번호(제○항), 규정 이름, 법칙 이름을 절대 쓰지 마세요. 어떤 규정에 해당하는지 판단하지 마세요.
2. 교정 전후에서 바뀐 부분만 짚어, 어떤 단어가 어떻게 바뀌었는지(띄어쓰기, 붙여쓰기, 글자 표기)만 적으세요.
3. 반드시 명사형으로 끝내세요. '~합니다', '~입니다', '~이다', '~해요'로 끝내지 마세요.
4. 한 줄로, 검색어만 출력하세요."""
REWRITE_PROMPT = "{query}"
REWRITE_FEWSHOT = [                          # 평가 문항과 무관한 예시
    ("친구 한명이 우리반에 → 친구 한 명이 우리 반에", "'한명'을 '한 명'으로, '우리반'을 '우리 반'으로 띄어쓰기"),
    ("내일 다시 올께 → 내일 다시 올게", "'올께'를 '올게'로 고친 표기"),
    ("숙제를 다해서 기분좋다 → 숙제를 다 해서 기분 좋다", "'다해서'를 '다 해서'로, '기분좋다'를 '기분 좋다'로 띄어쓰기"),
]
REWRITE_MAX_NEW_TOKENS, REWRITE_BATCH = 64, 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not os.path.exists(EXCEL_PATH):
    cand = [p for p in glob.glob(f"{BASE}/**/*.xlsx", recursive=True) if "_라벨" in unicodedata.normalize("NFC", p)]
    if not cand:
        raise FileNotFoundError(f"라벨 엑셀을 찾지 못했습니다: {BASE}")
    EXCEL_PATH = cand[0]
print("엑셀:", EXCEL_PATH)
print("device:", DEVICE, "| CTX:", CTX)
if DEVICE == "cpu":
    print("⚠ GPU가 아닙니다. '런타임 → 런타임 유형 변경 → T4 GPU' 후 셀 0부터 다시 실행하세요.")

def _md5(x):
    return hashlib.md5(json.dumps(x, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:10]

def _show(d, fmt):
    try:
        display(d.style.format(fmt, na_rep="-"))
    except Exception:
        print(d.to_string())

In [ ]:
# =====================================================================
# [셀 2] diff 추출 — 원문과 교정문을 어절 단위로 비교해 RAG 질의를 만든다
# =====================================================================
PUNCT_RE = re.compile(r"[.,!?~…·;:\"'“”‘’()\[\]{}<>「」『』\-–—/]+")

def _nfc(s):
    return unicodedata.normalize("NFC", str(s)).strip()

def _core(tokens):
    return PUNCT_RE.sub("", " ".join(tokens))

def _sp(s):
    """문장 부호 제거 + 공백 정리(띄어쓰기는 유지)"""
    return re.sub(r"\s+", " ", PUNCT_RE.sub("", _nfc(s))).strip()

def extract_diffs(src, cor, ctx=CTX, ignore_punct_only=True):
    """
    반환: [{'err','cor': 바뀐 어절, 'err_ctx','cor_ctx': 앞뒤 ctx어절 포함,
            'query': "err_ctx → cor_ctx", 'kind': '띄어쓰기'|'표기', 'i1','i2': 원문 어절 위치}]
    문장 부호만 바뀐 조각은 제외(et5는 문장 부호를 교정하지 않음)
    """
    a, b = _nfc(src).split(), _nfc(cor).split()
    out = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, a, b, autojunk=False).get_opcodes():
        if op == "equal":
            continue
        err_tok, cor_tok = a[i1:i2], b[j1:j2]
        if ignore_punct_only and _core(err_tok) == _core(cor_tok):
            continue
        L, R = max(i1 - ctx, 0), min(i2 + ctx, len(a))
        Lb, Rb = max(j1 - ctx, 0), min(j2 + ctx, len(b))
        err_ctx = re.sub(r"[.!?~…]+$", "", " ".join(a[L:R]))
        cor_ctx = re.sub(r"[.!?~…]+$", "", " ".join(b[Lb:Rb]))
        same = _core(err_tok).replace(" ", "") == _core(cor_tok).replace(" ", "")
        out.append({"err": " ".join(err_tok), "cor": " ".join(cor_tok), "err_ctx": err_ctx, "cor_ctx": cor_ctx,
                    "query": f"{err_ctx} → {cor_ctx}", "kind": "띄어쓰기" if same else "표기", "i1": i1, "i2": i2})
    return out

for s, c in [("오늘 학교에서 친구 한명이 우리반에 전학을 왔다", "오늘 학교에서 친구 한 명이 우리 반에 전학을 왔다"),
             ("학교 가는데 30분 걸려서 알게되서 다행이야.", "학교 가는 데 30분 걸려서 알게 돼서 다행이야")]:
    for d in extract_diffs(s, c):
        print(d["kind"], "|", d["query"])

In [ ]:
# =====================================================================
# [셀 3] 평가셋 로드 — '사용' 문항 중 N_SAMPLE개 층화 무작위 추출, 정답 조항 집합
# =====================================================================
df = pd.read_excel(EXCEL_PATH, sheet_name="검수", dtype=str)
df = df[df["사용 여부"].fillna("").str.strip() == "사용"].copy().reset_index(drop=True)
print("'사용' 문항:", len(df))

if N_SAMPLE and N_SAMPLE < len(df):
    strata = df.groupby(["추출 그룹", "평가 용도"]).size()
    quota = strata / strata.sum() * N_SAMPLE
    alloc = np.floor(quota).astype(int)
    for k in (quota - alloc).sort_values(ascending=False).index[:N_SAMPLE - alloc.sum()]:
        alloc[k] += 1
    df = pd.concat([g.sample(n=alloc[k], random_state=SAMPLE_SEED) for k, g in df.groupby(["추출 그룹", "평가 용도"])]
                  ).sort_values("문항ID").reset_index(drop=True)

df["gold"] = df.apply(lambda r: sorted({_nfc(x) for x in [r.get("정답 조항 1"), r.get("정답 조항 2")]
                                         if isinstance(x, str) and x.strip()}), axis=1)
assert (df["gold"].str.len() > 0).all()
df["src"] = df["오류 문장(맥락)"].map(_nfc)
df[["문항ID", "추출 그룹", "평가 용도"]].to_csv(f"{OUT_DIR}/eval_sample_{N_SAMPLE}_seed{SAMPLE_SEED}.csv",
                                          index=False, encoding="utf-8-sig")
print("평가 문항:", len(df), "| dev", (df["평가 용도"] == "dev").sum(), "/ test", (df["평가 용도"] == "test").sum())

In [ ]:
# =====================================================================
# [셀 4] et5 교정 (캐시)
# =====================================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

ET5_CACHE = f"{CACHE_DIR}/et5_{ET5_MODEL.split('/')[-1]}_b{ET5_BEAMS}_L{ET5_MAX_LEN}.json"
et5_cache = json.load(open(ET5_CACHE, encoding="utf-8")) if os.path.exists(ET5_CACHE) else {}
todo = [s for s in dict.fromkeys(df["src"]) if s not in et5_cache]
print(f"et5 교정 필요 {len(todo)}개 (캐시 {len(et5_cache)}개)")
if todo:
    et5_tok = AutoTokenizer.from_pretrained(ET5_MODEL)
    et5 = AutoModelForSeq2SeqLM.from_pretrained(ET5_MODEL).to(DEVICE).eval()
    for s in range(0, len(todo), ET5_BATCH):
        batch = todo[s:s + ET5_BATCH]
        enc = et5_tok([ET5_PREFIX + x for x in batch], return_tensors="pt", padding=True,
                      truncation=True, max_length=ET5_MAX_LEN).to(DEVICE)
        with torch.no_grad():
            gen = et5.generate(**enc, max_length=ET5_MAX_LEN, num_beams=ET5_BEAMS, early_stopping=True)
        for x, y in zip(batch, et5_tok.batch_decode(gen, skip_special_tokens=True)):
            et5_cache[x] = _nfc(y)
        json.dump(et5_cache, open(ET5_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"et5 교정 {min(s + ET5_BATCH, len(todo))}/{len(todo)}")
    del et5, et5_tok; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
df["et5"] = df["src"].map(et5_cache)

In [ ]:
# =====================================================================
# [셀 5] diff(원문 vs et5) → 질의 + et5 교정 상태
#   이 문항의 '틀린 말'(C열) 위치와 겹치는 diff 조각만 사용
#   일치  = 정답('고친 말')대로 고침 → RAG 비교의 주 평가 대상
#   다르게 = 그 자리를 다르게 고침(참고용) / 누락 = 고치지 않음(질의 없음)
# =====================================================================
def locate(tokens, target):
    tgt = _sp(target).replace(" ", "")
    if not tgt:
        return None
    flat, owner = "", []
    for i, t in enumerate(tokens):
        c = PUNCT_RE.sub("", t); flat += c; owner += [i] * len(c)
    k = flat.find(tgt)
    return None if k < 0 else (owner[k], owner[k + len(tgt) - 1] + 1)

def match_segment(row, ctx=CTX):
    span = locate(row["src"].split(), row["틀린 말"])
    if span is None:
        return None, "위치 못 찾음"
    s, e = span
    segs = extract_diffs(row["src"], row["et5"], ctx=ctx)
    over = [d for d in segs if (d["i1"] < e and d["i2"] > s) or (d["i1"] == d["i2"] and s <= d["i1"] <= e)]
    if not over:
        return None, "누락"
    gold_fix = _sp(row["고친 말"])
    for d in over:
        if gold_fix and gold_fix in _sp(d["cor_ctx"]):
            return d, "일치"
    return over[0], "다르게"

segs = [match_segment(r) for _, r in df.iterrows()]
df["seg"] = [d for d, _ in segs]
df["et5_status"] = [st for _, st in segs]
df["query"] = [d["query"] if d else None for d in df["seg"]]
print("et5 교정 상태:", df["et5_status"].value_counts().to_dict())
print(pd.crosstab(df["추출 그룹"], df["et5_status"], margins=True, margins_name="합계"))

In [ ]:
# =====================================================================
# [셀 6] et5 성능 검증 — 정답 교정(H열) 대비, 문장 길이 구간별
# =====================================================================
def _norm(s):
    return re.sub(r"\s+", " ", PUNCT_RE.sub("", _nfc(s))).strip()

def _lev(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb)))
        prev = cur
    return prev[-1]

ev = pd.DataFrame({"src": df["src"].map(_norm), "et5": df["et5"].map(_norm), "gold": df["교정 문장"].map(_norm),
                   "추출 그룹": df["추출 그룹"]})
ev["길이 구간"] = pd.cut(ev["src"].str.len(), [0, 20, 40, 60, 100, 10_000],
                      labels=["~20자", "21~40자", "41~60자", "61~100자", "101자~"])
ev["오류(원문)"] = [_lev(s, g) for s, g in zip(ev["src"], ev["gold"])]
ev["오류(et5)"] = [_lev(e, g) for e, g in zip(ev["et5"], ev["gold"])]
ev["정답대로"] = (df["et5_status"] == "일치").astype(int).values

def et5_summary(g):
    return pd.Series({"문항 수": len(g), "완전 일치율": (g["et5"] == g["gold"]).mean(),
                      "오류 감소율": 1 - g["오류(et5)"].sum() / max(g["오류(원문)"].sum(), 1),
                      "CER(원문)": g["오류(원문)"].sum() / g["gold"].str.len().sum(),
                      "CER(et5)": g["오류(et5)"].sum() / g["gold"].str.len().sum(),
                      "안 고침": (g["et5"] == g["src"]).mean(),
                      "잘림 의심": (g["et5"].str.len() < g["src"].str.len() * 0.8).mean(),
                      "이 문항 오류 정답대로": g["정답대로"].mean()})
fmt = {c: "{:.1%}" for c in ["완전 일치율", "오류 감소율", "CER(원문)", "CER(et5)", "안 고침", "잘림 의심", "이 문항 오류 정답대로"]}
fmt["문항 수"] = "{:.0f}"
_show(et5_summary(ev).to_frame("전체").T, fmt)
_show(ev.groupby("길이 구간", observed=True).apply(et5_summary), fmt)

In [ ]:
# =====================================================================
# [셀 7] 맞춤법 본문 로드 → 조항 단위 분할 (부록 '문장 부호'는 제외)
#   HTML의 굵게·강조 태그를 벗겨야 예문 띄어쓰기가 깨지지 않는다('아는 것이' → '아는 것 이' 방지)
# =====================================================================
FORCE_REDOWNLOAD = False

if FORCE_REDOWNLOAD or not os.path.exists(RULES_TXT):
    import requests
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(requests.get(RULES_URL, timeout=30, headers={"User-Agent": "Mozilla/5.0"}).text, "html.parser")
    for t in soup(["script", "style", "nav", "header", "footer"]):
        t.decompose()
    for t in soup.find_all(["b", "strong", "em", "i", "u", "span", "font", "sup", "sub", "a", "mark"]):
        t.unwrap()
    for br in soup.find_all("br"):
        br.replace_with("\n")
    soup.smooth()
    open(RULES_TXT, "w", encoding="utf-8").write(re.sub(r"\n\s*\n+", "\n", soup.get_text("\n")))
    print("본문을 받아 저장:", RULES_TXT)

raw = _nfc(open(RULES_TXT, encoding="utf-8").read())

def split_articles(text):
    """줄 맨 앞 '제N항'을 1→57 순서대로만 경계로 인정(해설 속 인용 무시), 마지막 조항 뒤 부록 분리"""
    marks, expect = [], 1
    for m in re.finditer(r"(?m)^\s*제\s*(\d+)\s*항", text):
        if int(m.group(1)) == expect:
            marks.append((expect, m.start())); expect += 1
        if expect > 57:
            break
    assert len(marks) >= 50, f"조항 경계를 {len(marks)}개만 찾았습니다."
    arts = [("맞춤법-0", text[:marks[0][1]])]
    for k, (n, st) in enumerate(marks):
        body = text[st: marks[k+1][1] if k + 1 < len(marks) else len(text)]
        m = re.search(r"(?m)^\s*\[?\s*부\s*록", body) if k + 1 == len(marks) else None
        if m:
            arts += [(f"맞춤법-{n}", body[:m.start()]), ("맞춤법-부록", body[m.start():])]
        else:
            arts.append((f"맞춤법-{n}", body))
    return [(lab, re.sub(r"[ \t]+", " ", t).strip()) for lab, t in arts if t.strip()]

ARTICLES = [(l, t) for l, t in split_articles(raw) if l != "맞춤법-부록"]
print("조항 수:", len(ARTICLES), "(정상 58) | 총 글자:", sum(len(t) for _, t in ARTICLES))
for s in ["아는 것이 힘이다", "아는 것 이 힘이다"]:
    print(f"  '{s}' 포함: {s in raw}")      # True / False 여야 정상

In [ ]:
# =====================================================================
# [셀 8] 청킹 2종 + 청크 대표 조항 라벨
#   고정500: 500자씩 자름, 대표 조항 = 청크 안에서 글자 수가 가장 많은 조항
#   구조+semantic: '제N항' 경계에서 반드시 자르고, 조항 안에서는
#     이웃 문장 유사도가 문서 전체 하위 SEM_PCT% 이하인 지점(청크 200자 이상일 때)에서 자르며 800자를 넘기지 않음
# =====================================================================
from sentence_transformers import SentenceTransformer, CrossEncoder

def chunk_fixed(articles, size=FIXED_LEN):
    full, owner = "", []
    for lab, t in articles:
        full += t + "\n"; owner += [lab] * (len(t) + 1)
    chunks = []
    for s in range(0, len(full), size):
        txt = full[s:s+size].strip()
        if txt:
            cnt = Counter(owner[s:s+size])
            chunks.append({"text": txt, "label": cnt.most_common(1)[0][0], "labels": sorted(cnt)})
    return chunks

def _sentences(t):
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n+", t) if p.strip()]

def chunk_struct_semantic(articles, model, pct=SEM_PCT, mn=SEM_MIN, mx=SEM_MAX):
    per_art = []
    for lab, t in articles:
        sents = _sentences(t)
        if len(sents) >= 2:
            E = model.encode(sents, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
            sim = (E[:-1] * E[1:]).sum(1)
        else:
            sim = np.array([])
        per_art.append((lab, sents, sim))
    thr = np.percentile(np.concatenate([s for _, _, s in per_art if len(s)]), pct)
    chunks = []
    for lab, sents, sim in per_art:
        cur, cur_len = [], 0
        def flush():
            nonlocal cur, cur_len
            if cur:
                chunks.append({"text": " ".join(cur), "label": lab, "labels": [lab]})
            cur, cur_len = [], 0
        for i, s in enumerate(sents):
            while len(s) > mx:
                flush(); chunks.append({"text": s[:mx], "label": lab, "labels": [lab]}); s = s[mx:]
            if cur and cur_len + len(s) > mx:
                flush()
            cur.append(s); cur_len += len(s) + 1
            if i < len(sim) and sim[i] <= thr and cur_len >= mn:
                flush()
        if cur:
            if chunks and chunks[-1]["label"] == lab and cur_len < mn and len(chunks[-1]["text"]) + cur_len + 1 <= mx:
                chunks[-1]["text"] += " " + " ".join(cur)
            else:
                flush()
    print(f"semantic 경계 기준값(하위 {pct}% 유사도): {thr:.3f}")
    return chunks

_sem = SentenceTransformer(SEM_SPLIT_MODEL, device=DEVICE)
if DEVICE == "cuda": _sem.half()
CHUNKS = {"고정500": chunk_fixed(ARTICLES), "구조+semantic": chunk_struct_semantic(ARTICLES, _sem)}
del _sem; gc.collect()
if DEVICE == "cuda": torch.cuda.empty_cache()
for k, v in CHUNKS.items():
    L = np.array([len(c["text"]) for c in v])
    print(f"{k}: {len(v)}개, 평균 {L.mean():.0f}자, 최대 {L.max()}자, 조항 걸침 {sum(len(c['labels']) > 1 for c in v)}개")

In [ ]:
# =====================================================================
# [셀 9] 쿼리 재작성(kanana 2.1B, 캐시) — '재작성 있음' 조합용
# =====================================================================
REWRITE_CACHE = f"{CACHE_DIR}/rewrite_{REWRITE_MODEL.split('/')[-1]}_{_md5([REWRITE_SYSTEM, REWRITE_PROMPT, REWRITE_FEWSHOT])}.json"
rw_cache = json.load(open(REWRITE_CACHE, encoding="utf-8")) if os.path.exists(REWRITE_CACHE) else {}
todo = [q for q in dict.fromkeys(df["query"].dropna()) if q not in rw_cache]
print(f"재작성 필요 {len(todo)}개 (캐시 {len(rw_cache)}개)")
ART_RE = re.compile(r"(한글\s*맞춤법\s*)?제\s*\d+\s*항(\s*의?\s*\d+)?(에\s*따르면|에\s*따라|에서는|에\s*의하면)?[\s,]*")

def clean_rw(o, q):
    o = o.strip().split("\n")[0].strip().strip('"').strip("'")
    o = re.sub(r"^(검색어|출력)\s*[:：]\s*", "", o)
    o = ART_RE.sub("", o).strip(" ,.")
    return o or q

if todo:
    tok = AutoTokenizer.from_pretrained(REWRITE_MODEL); tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    llm = AutoModelForCausalLM.from_pretrained(
        REWRITE_MODEL, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32).to(DEVICE).eval()
    def _prompt(q):
        msgs = [{"role": "system", "content": REWRITE_SYSTEM}]
        for x, y in REWRITE_FEWSHOT:
            msgs += [{"role": "user", "content": REWRITE_PROMPT.format(query=x)}, {"role": "assistant", "content": y}]
        msgs.append({"role": "user", "content": REWRITE_PROMPT.format(query=q)})
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    for s in range(0, len(todo), REWRITE_BATCH):
        batch = todo[s:s + REWRITE_BATCH]
        enc = tok([_prompt(q) for q in batch], return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            gen = llm.generate(**enc, max_new_tokens=REWRITE_MAX_NEW_TOKENS, do_sample=False,
                               pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for q, o in zip(batch, tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            rw_cache[q] = {"raw": o.strip(), "clean": clean_rw(o, q)}
        json.dump(rw_cache, open(REWRITE_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"재작성 {min(s + REWRITE_BATCH, len(todo))}/{len(todo)}")
    del llm, tok; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()

df["query_rw"] = [rw_cache[q]["clean"] if isinstance(q, str) and q in rw_cache else q for q in df["query"]]

In [ ]:
# =====================================================================
# [셀 10] 질의 보강(1차 추가 시도) — diff 조각을 형태소 분석해 문법 범주를 질의 앞에 붙임
#   예) "공부를 안하는게 → 공부를 안 하는 게"
#    →  "단어 '하' 앞 띄어쓰기; 의존 명사 '것' 띄어쓰기 | 공부를 안하는게 → 공부를 안 하는 게"
#   규칙 기반(LLM 아님)이고 조항 번호는 쓰지 않는다.
# =====================================================================
from kiwipiepy import Kiwi
kiwi = Kiwi()

_UNIT = set("개 명 번 살 년 월 일 시 분 초 시간 달 주 개월 배 권 장 대 마리 잔 병 가지 군데 층 학년 차 회 벌 채 켤레 강 판 그루 송이 척 쪽 원 킬로 미터 센티".split())
_NUM_MM = set("한 두 세 네 다섯 여섯 일곱 여덟 아홉 열 몇 스무 서 너 여러 수십 수백 첫".split())
_LIST = set("겸 내지 대 등 및 등등 등속 등지".split())
_JONG = " ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ"

def _ns(s):
    return PUNCT_RE.sub("", _nfc(s)).replace(" ", "")

def _bounds(s):
    s = PUNCT_RE.sub("", _nfc(s)); b, k = set(), 0
    for ch in s:
        if ch == " ": b.add(k)
        else: k += 1
    return b

def _toks(s):
    s = PUNCT_RE.sub("", _nfc(s)); m, k = [], 0
    for ch in s:
        m.append(k)
        if ch != " ": k += 1
    m.append(k)
    return [(m[t.start], t.form, t.tag) for t in kiwi.tokenize(s)]

def _jong(ch):
    o = ord(ch) - 0xAC00
    return _JONG[o % 28] if 0 <= o < 11172 else ""

def _shift_typo(x, y):
    """ㅅ/ㅆ, ㄷ/ㄸ, ㅂ/ㅃ, ㄱ/ㄲ, ㅈ/ㅉ, ㅐ/ㅒ, ㅔ/ㅖ만 다른 한 글자(Shift 오타)"""
    if len(x) != 1 or len(y) != 1: return False
    ox, oy = ord(x) - 0xAC00, ord(y) - 0xAC00
    if not (0 <= ox < 11172 and 0 <= oy < 11172): return False
    ax, ay = (ox // 588, (ox % 588) // 28, ox % 28), (oy // 588, (oy % 588) // 28, oy % 28)
    d = [i for i in range(3) if ax[i] != ay[i]]
    if len(d) != 1: return False
    i = d[0]; p = tuple(sorted((ax[i], ay[i])))
    return ((i == 0 and p in {(9, 10), (3, 4), (7, 8), (0, 1), (12, 13)}) or
            (i == 2 and p in {(19, 20), (1, 2)}) or (i == 1 and p in {(1, 3), (5, 7)}))

def gram_features(err, cor, max_feats=4):
    feats = []
    e, c = _ns(err), _ns(cor)
    sm = difflib.SequenceMatcher(None, e, c, autojunk=False)
    c2e = {b + t: a + t for a, b, size in sm.get_matching_blocks() for t in range(size)}
    be, bc = _bounds(err), _bounds(cor)
    T = _toks(cor); start = {}
    for i, t in enumerate(T): start.setdefault(t[0], i)
    for k in sorted(bc):                                    # 새로 띄운 경계
        if k in c2e and c2e[k] in be: continue
        i = start.get(k)
        if i is None: continue
        _, f, tg = T[i]; p = T[i-1] if i > 0 else None
        if f in _LIST: feats.append(f"열거하는 말 '{f}' 띄어쓰기")
        elif (tg == "NNB" or f in _UNIT) and p and (p[2] in ("SN", "NR") or p[1] in _NUM_MM):
            feats.append(f"수 뒤 단위 명사 '{f}' 띄어쓰기")
        elif tg == "NNB" or f in ("거", "걸", "건", "게"): feats.append(f"의존 명사 '{f}' 띄어쓰기")
        elif tg == "VX": feats.append(f"보조 용언 '{f}' 띄어쓰기")
        elif tg.startswith("J"): feats.append(f"조사 '{f}' 앞 띄어쓰기")
        elif tg.startswith("XS"): feats.append(f"접미사 '{f}' 앞 띄어쓰기")
        else: feats.append(f"단어 '{f}' 앞 띄어쓰기")
    inv = {v: k for k, v in c2e.items()}
    for j in sorted(be):                                    # 붙인 경계
        k = inv.get(j)
        if k is None or k in bc: continue
        i = start.get(k)
        if i is None: continue
        _, f, tg = T[i]
        if tg.startswith("J") or tg == "VCP": feats.append(f"조사 '{f}' 붙여 쓰기")
        elif tg.startswith("XS") or tg.startswith("E"): feats.append(f"접미사·어미 '{f}' 붙여 쓰기")
        elif tg == "VX": feats.append(f"보조 용언 '{f}' 붙여 쓰기")
        else: feats.append(f"'{f}' 앞 붙여 쓰기(한 단어)")
    for op, a1, a2, b1, b2 in sm.get_opcodes():             # 글자 표기 변화
        if op == "equal": continue
        x, y = e[a1:a2], c[b1:b2]
        if _shift_typo(x, y): continue
        wy = c[max(b1-2, 0):b2+2]
        if ("돼" in x + y or "됬" in x or "됀" in x): feats.append("'되'와 '돼' 구별, '되어'가 줄어든 '돼' 표기")
        elif "않" in y or ("안" in y and "않" in x): feats.append("'않다'(아니하다)와 부사 '안' 구별 표기")
        elif "잖" in y or "찮" in y: feats.append("'-지 않-'이 줄어든 '-잖-', '-하지 않-'이 줄어든 '-찮-' 표기")
        elif any(s in x for s in ("께", "껄", "쑤록", "찌")) and any(s in y for s in ("게", "걸", "수록", "지")):
            feats.append(f"어미 '{y}' 예사소리 표기(-ㄹ게, -ㄹ걸, -ㄹ수록, -ㄹ지)")
        elif x == "가" and y == "까" and b1 > 0 and _jong(c[b1-1]) == "ㄹ" and c[b2:b2+1] != "지":
            feats.append("의문 어미 '-ㄹ까' 된소리 표기")
        elif ("히" in y and "이" in x) or ("이" in y and "히" in x): feats.append(f"부사 끝 '-이'와 '-히' 구별 표기('{wy}')")
        elif ("더" in y and "드" in x) or ("던" in y and "든" in x) or ("든" in y and "던" in x):
            feats.append("지난 일 '-더라, -던'과 선택 '-든' 구별 표기")
        elif "로서" in wy or "로써" in wy: feats.append("'로서'와 '로써' 구별 표기")
        elif "의" in y and "이" in x: feats.append(f"'ㅢ' 표기('{wy}')")
        elif "며칠" in wy: feats.append("'며칠' 표기")
        else:
            stem = next((t for t in kiwi.tokenize(c[max(b1-3, 0):b2+3]) if t.tag in ("VV", "VA", "VX")), None)
            feats.append(f"용언 '{stem.form}다'의 어간과 어미를 밝혀 적은 표기('{wy}')" if stem else f"'{x}'를 '{y}'로 고친 표기")
    return list(dict.fromkeys(feats))[:max_feats]

df["gram_feats"] = [gram_features(d["err"], d["cor"]) if d else None for d in df["seg"]]
df["query_enr"] = [("; ".join(f) + " | " if f else "") + q if isinstance(q, str) else None
                   for f, q in zip(df["gram_feats"], df["query"])]
print(df.loc[df["query_enr"].notna(), ["query", "query_enr"]].head(5).to_string(index=False))

In [ ]:
# =====================================================================
# [셀 11] 모델 로드 + 벡터DB + BM25 + 검색 함수 (한 번만 실행)
#   벡터만: 코사인 유사도 순
#   하이브리드+리랭킹: 벡터 50 + BM25(kiwi) 50 → RRF 20 → bge-reranker-v2-m3(크로스 인코더) 재정렬
# =====================================================================
from rank_bm25 import BM25Okapi

def ktok(s):
    return [t.form for t in kiwi.tokenize(s.replace("→", " ")) if not t.tag.startswith("S") or t.tag in ("SL", "SN", "SH")]

if "MODELS" not in globals():
    MODELS = {}
for name, path in EMBED_MODELS.items():
    if name not in MODELS:
        m = SentenceTransformer(path, device=DEVICE)
        if DEVICE == "cuda": m.half()
        MODELS[name] = m
if "reranker" not in MODELS:
    rr = CrossEncoder(RERANK_MODEL, max_length=512, device=DEVICE)
    if DEVICE == "cuda": rr.model.half()
    MODELS["reranker"] = rr

CHUNK_EMB, BM25 = {}, {}
def build_index(ck):
    for emb in EMBED_MODELS:
        CHUNK_EMB[(emb, ck)] = MODELS[emb].encode([c["text"] for c in CHUNKS[ck]], normalize_embeddings=True,
                                                 batch_size=16, show_progress_bar=False,
                                                 convert_to_numpy=True).astype(np.float32)
    BM25[ck] = BM25Okapi([ktok(c["text"]) for c in CHUNKS[ck]])
for ck in CHUNKS:
    build_index(ck)

def search_all(queries, emb, chunker, method):
    Q = MODELS[emb].encode(list(queries), normalize_embeddings=True, batch_size=32,
                           show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
    S = Q @ CHUNK_EMB[(emb, chunker)].T
    if method == "벡터만":
        return [[int(i) for i in np.argsort(-s)[:max(TOPK_EVAL, 10)]] for s in S]
    cand = []
    for qi, q in enumerate(queries):
        rrf = defaultdict(float)
        for r, i in enumerate(np.argsort(-S[qi])[:VEC_TOPN]): rrf[int(i)] += 1 / (RRF_K + r + 1)
        for r, i in enumerate(np.argsort(-BM25[chunker].get_scores(ktok(q)))[:BM25_TOPN]): rrf[int(i)] += 1 / (RRF_K + r + 1)
        cand.append([i for i, _ in sorted(rrf.items(), key=lambda x: -x[1])[:RRF_TOPN]])
    texts = [c["text"] for c in CHUNKS[chunker]]
    scores = MODELS["reranker"].predict([(q, texts[i]) for q, cl in zip(queries, cand) for i in cl],
                                        batch_size=64, show_progress_bar=False)
    scores = np.nan_to_num(np.asarray(scores, dtype=np.float32), nan=-1e9)
    out, p = [], 0
    for cl in cand:
        sc = scores[p:p+len(cl)]; p += len(cl)
        out.append([cl[j] for j in np.argsort(-sc)])
    return out

has_q = df["query"].notna().values
q_idx = np.where(has_q)[0]

def run_search(qcol, emb, chunker, method, tag):
    """검색 결과를 Drive에 저장하고, 같은 조건이면 저장본을 재사용"""
    sig = _md5({"q": df.loc[has_q, ["문항ID", qcol]].values.tolist(), "emb": EMBED_MODELS[emb], "method": method,
                "chunks": _md5([c["text"] for c in CHUNKS[chunker]]),
                "p": [VEC_TOPN, BM25_TOPN, RRF_K, RRF_TOPN, RERANK_MODEL]})
    path = f"{RANK_DIR}/{tag}_{sig}.json"
    t = time.time()
    if os.path.exists(path):
        rq, msg = json.load(open(path)), "저장된 결과"
    else:
        rq = search_all(df.loc[has_q, qcol].tolist(), emb, chunker, method)
        json.dump(rq, open(path, "w")); msg = f"{time.time()-t:.0f}s"
    ranks = [None] * len(df)
    for i, rk in zip(q_idx, rq): ranks[i] = rk
    return ranks, msg

for emb in EMBED_MODELS:                                   # 스모크 테스트
    for method in ["벡터만", "하이브리드+리랭킹"]:
        r = search_all(df["query"].dropna().head(3).tolist(), emb, "구조+semantic", method)
        print(f"테스트 OK: {emb} | {method} → {[CHUNKS['구조+semantic'][i]['label'] for i in r[0][:3]]}")

In [ ]:
# =====================================================================
# [셀 12] 1차 실험: 16개 조합 (베이스라인 #1 = 재작성 있음 · bge-m3 · 벡터만 · 고정500)
# =====================================================================
BASE_CFG = {"쿼리": "재작성 있음", "임베딩": "bge-m3", "검색": "벡터만", "청킹": "고정500"}
ALT = {"쿼리": "재작성 없음", "임베딩": "KURE", "검색": "하이브리드+리랭킹", "청킹": "구조+semantic"}
AXES = list(BASE_CFG)
ORDER = [(), ("쿼리",), ("임베딩",), ("검색",), ("청킹",),
         ("쿼리", "임베딩"), ("쿼리", "검색"), ("쿼리", "청킹"), ("임베딩", "검색"), ("임베딩", "청킹"), ("검색", "청킹"),
         ("쿼리", "임베딩", "검색"), ("쿼리", "임베딩", "청킹"), ("쿼리", "검색", "청킹"), ("임베딩", "검색", "청킹"),
         ("쿼리", "임베딩", "검색", "청킹")]
COMBOS = []
for n, changed in enumerate(ORDER, 1):
    cfg = {a: (ALT[a] if a in changed else BASE_CFG[a]) for a in AXES}
    cfg.update({"#": n, "바꾼 것": ", ".join(changed) or "없음(베이스라인)"})
    COMBOS.append(cfg)

RANKS = {}
for cfg in sorted(COMBOS, key=lambda c: (c["임베딩"], c["청킹"], c["검색"], c["쿼리"])):
    qcol = "query_rw" if cfg["쿼리"] == "재작성 있음" else "query"
    RANKS[cfg["#"]], msg = run_search(qcol, cfg["임베딩"], cfg["청킹"], cfg["검색"], f"combo{cfg['#']:02d}")
    print(f"#{cfg['#']:>2} {cfg['쿼리']} | {cfg['임베딩']} | {cfg['검색']} | {cfg['청킹']}  ({msg})")

In [ ]:
# =====================================================================
# [셀 13] 지표 — 엄격 기준: 검색된 청크의 '대표 조항'이 정답 집합에 있으면 정답
#   주 지표는 et5 '일치' 문항(test). 전체 성공률은 평가 문항 전체 기준(et5 실패 포함)
# =====================================================================
def item_scores(ranks, chunker):
    labs = [c["label"] for c in CHUNKS[chunker]]
    r1, r3, mrr = [], [], []
    for rk, gold in zip(ranks, df["gold"]):
        if rk is None:
            r1.append(0); r3.append(0); mrr.append(0.0); continue
        hit = next((p for p, l in enumerate(labs[i] for i in rk[:TOPK_EVAL]) if l in gold), None)
        r1.append(int(hit == 0)); r3.append(int(hit is not None)); mrr.append(0.0 if hit is None else 1 / (hit + 1))
    return np.array(r1), np.array(r3), np.array(mrr)

def _mean(x):
    return float(np.mean(x)) if len(x) else np.nan

def mcnemar_p(a, b):
    n01, n10 = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    n = n01 + n10
    return 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, i) for i in range(min(n01, n10) + 1)) / 2 ** n)

def boot_ci(x, B=2000, seed=0):
    if len(x) == 0: return (np.nan, np.nan)
    m = x[np.random.default_rng(seed).integers(0, len(x), size=(B, len(x)))].mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

is_dev = (df["평가 용도"] == "dev").values
ok = (df["et5_status"] == "일치").values
test_ok = ~is_dev & ok
no_art2 = df["gold"].apply(lambda g: "맞춤법-2" not in g).values
MIN_ART_N = 5

def metric_row(ranks, chunker, base_r3=None):
    r1, r3, mrr = item_scores(ranks, chunker)
    hits = defaultdict(list)
    for g, h in zip(df["gold"].values[test_ok], r3[test_ok]):
        for a in g: hits[a].append(h)
    lo, hi = boot_ci(r3[test_ok])
    return {"dev R@3": _mean(r3[is_dev & ok]), "test R@1": _mean(r1[test_ok]), "test R@3": _mean(r3[test_ok]),
            "test R@3 95%CI": f"{lo:.3f}~{hi:.3f}", "test MRR@3": _mean(mrr[test_ok]),
            "test R@3(제2항 제외)": _mean(r3[test_ok & no_art2]),
            "test R@3(조항 평균)": _mean([np.mean(v) for v in hits.values() if len(v) >= MIN_ART_N]),
            "test 전체 성공률@3": _mean((r3 * ok)[~is_dev]),
            "vs 기준 p값": np.nan if base_r3 is None else mcnemar_p(r3[test_ok], base_r3[test_ok])}, r3, hits

base_r3 = item_scores(RANKS[1], "고정500")[1]
rows, per_item, per_art = [], {}, {}
for cfg in COMBOS:
    row, r3, hits = metric_row(RANKS[cfg["#"]], cfg["청킹"], None if cfg["#"] == 1 else base_r3)
    per_item[cfg["#"]], per_art[cfg["#"]] = r3, hits
    rows.append({"#": cfg["#"], "바꾼 것": cfg["바꾼 것"], **{a: cfg[a] for a in AXES}, **row})
res = pd.DataFrame(rows).set_index("#")
res["Δtest R@3"] = res["test R@3"] - res.loc[1, "test R@3"]

print("[et5 교정 상태]"); print(pd.crosstab(df["평가 용도"], df["et5_status"], margins=True, margins_name="합계"))
print(f"\nRAG 비교 문항(et5 '일치'): dev {(is_dev & ok).sum()} / test {test_ok.sum()}")
show = ["바꾼 것", "dev R@3", "test R@1", "test R@3", "test R@3 95%CI", "Δtest R@3", "vs 기준 p값",
        "test MRR@3", "test R@3(제2항 제외)", "test R@3(조항 평균)", "test 전체 성공률@3"]
_show(res[show], {c: "{:.3f}" for c in show if c not in ("바꾼 것", "test R@3 95%CI")})

eff = pd.DataFrame([{"축": a, "베이스라인 쪽": BASE_CFG[a], "바꾼 쪽": ALT[a],
                     **{f"{m} 변화": res.loc[res[a] == ALT[a], m].mean() - res.loc[res[a] == BASE_CFG[a], m].mean()
                        for m in ["test R@3", "test R@1", "test MRR@3"]}} for a in AXES]).set_index("축")
print("\n[축별 평균 효과] (+면 '바꾼 쪽'이 더 좋음)")
_show(eff, {c: "{:+.3f}" for c in eff.columns if "변화" in c})

arts = sorted({a for d in per_art.values() for a, v in d.items() if len(v) >= MIN_ART_N}, key=lambda x: int(x.split("-")[1]))
art_tbl = pd.DataFrame({f"#{n}": {a: _mean(per_art[n].get(a, [])) for a in arts} for n in res.index})
art_tbl.insert(0, "문항 수", [len(per_art[1].get(a, [])) for a in arts])
print("\n[조항별 test recall@3 — 문항 5개 이상 조항]")
_show(art_tbl, {c: "{:.2f}" for c in art_tbl.columns if c != "문항 수"})

In [ ]:
# =====================================================================
# [셀 14] 원인 진단 — 파이프라인 점검, 1위 조항 쏠림, 라벨 단위(조항/절/장)별 recall
# =====================================================================
sanity = [("의존 명사는 띄어 쓴다", "맞춤법-42"), ("단위를 나타내는 명사는 띄어 쓴다", "맞춤법-43"),
          ("보조 용언은 띄어 씀을 원칙으로 한다", "맞춤법-47"), ("조사는 그 앞말에 붙여 쓴다", "맞춤법-41"),
          ("문장의 각 단어는 띄어 씀을 원칙으로 한다", "맞춤법-2"), ("되어 돼 준말 표기", "맞춤법-35")]
print("[규정 문장으로 검색 → 1위가 정답이면 파이프라인 정상]")
for emb in EMBED_MODELS:
    for ck in ["고정500", "구조+semantic"]:
        labs = [c["label"] for c in CHUNKS[ck]]
        r = search_all([q for q, _ in sanity], emb, ck, "벡터만")
        print(f"  {emb:6s} | {ck:12s} | {sum(labs[rk[0]] == g for rk, (_, g) in zip(r, sanity))}/{len(sanity)}")

for n in [4]:
    labs = [c["label"] for c in CHUNKS[COMBOS[n-1]["청킹"]]]
    top1 = Counter(labs[RANKS[n][i][0]] for i in np.where(test_ok)[0])
    print(f"\n{n}번 조합 1위 조항 분포 Top 6:", ", ".join(f"{k}:{v}" for k, v in top1.most_common(6)))

amap = pd.read_excel(EXCEL_PATH, sheet_name="조항 목록", dtype=str).dropna(subset=["조항 ID(라벨에 입력)"])
amap["장"] = amap["장"].fillna("").str.strip()
SEC = dict(zip(amap["조항 ID(라벨에 입력)"], (amap["장"] + " " + amap["절"].fillna("")).str.strip()))
CHP = dict(zip(amap["조항 ID(라벨에 입력)"], amap["장"]))

def recall3_level(ranks, chunker, f):
    labs = [c["label"] for c in CHUNKS[chunker]]
    return np.array([0 if rk is None else int(any(f(labs[i]) in {f(a) for a in g} for i in rk[:TOPK_EVAL]))
                     for rk, g in zip(ranks, df["gold"])])

lv = []
for cfg in COMBOS:
    row = {"#": cfg["#"], "바꾼 것": cfg["바꾼 것"]}
    for name, f in [("조항", lambda a: a), ("절", lambda a: SEC.get(a, a)), ("장", lambda a: CHP.get(a, a))]:
        r = recall3_level(RANKS[cfg["#"]], cfg["청킹"], f)
        row[f"R@3({name})"] = r[test_ok].mean(); row[f"R@3({name}, 제2항 제외)"] = r[test_ok & no_art2].mean()
    lv.append(row)
lvl = pd.DataFrame(lv).set_index("#")
print("\n[라벨 단위별 test recall@3]")
_show(lvl, {c: "{:.3f}" for c in lvl.columns if c != "바꾼 것"})

In [ ]:
# =====================================================================
# [셀 15] 1차 추가 시도: 질의 보강(셀 10) · 청크 머리말(모든 청크 앞에 조항 첫 문장)
#   기준: 4번 조합 (kanana 쿼리 재작성 · bge-m3 · 하이브리드 검색 · 고정 길이 500 청킹)
# =====================================================================
def _first_sent(t):
    m = re.search(r"^(.{5,120}?[.다])(\s|$)", t.strip())
    return (m.group(1) if m else t.strip()[:80]).strip()
ART_HEAD = {lab: _first_sent(t) for lab, t in ARTICLES}
for ck in ["고정500"]:
    hk = ck + "+머리말"
    if hk not in CHUNKS:
        CHUNKS[hk] = [{**c, "text": f"{ART_HEAD.get(c['label'], '')} | {c['text']}"} for c in CHUNKS[ck]]
        build_index(hk)

n, ck = 4, "고정500"
EXPS = [("4번 조합 기준", n, "query_rw", ck),
        ("4번 조합 + 형태소 질의 보강", n, "query_enr", ck),
        ("4번 조합 + 청크 머리말 달기", n, "query_rw", ck + "+머리말"),
        ("4번 조합 + 형태소 질의 보강 + 청크 머리말 달기", n, "query_enr", ck + "+머리말")]
EXP_RANKS, rows = {}, []
for name, n, qcol, ck in EXPS:
    if qcol == "query_rw" and "+머리말" not in ck:
        EXP_RANKS[name], msg = RANKS[n], "1차 실험 결과"
    else:
        EXP_RANKS[name], msg = run_search(qcol, "bge-m3", ck, "하이브리드+리랭킹", "exp")
    print(f"{name}  ({msg})")
    b3 = None if "기준" in name else item_scores(RANKS[n], ck.replace("+머리말", ""))[1]
    row, _, _ = metric_row(EXP_RANKS[name], ck, b3)
    rows.append({"실험": name, **row})
exp_res = pd.DataFrame(rows).set_index("실험")
print("\n[1차 추가 시도 결과 (test)]")
_show(exp_res, {c: "{:.3f}" for c in exp_res.columns if c != "test R@3 95%CI"})

In [ ]:
# =====================================================================
# [셀 16] 저장 — 결과표와 문항별 상세(오답 분석용). detail 파일에는 AI허브 문장이 있으니 저장소에 올리지 않는다
# =====================================================================
stamp = time.strftime("%Y%m%d_%H%M")
res.to_csv(f"{OUT_DIR}/rag16_summary_{stamp}.csv", encoding="utf-8-sig")
eff.to_csv(f"{OUT_DIR}/rag16_axis_effect_{stamp}.csv", encoding="utf-8-sig")
art_tbl.to_csv(f"{OUT_DIR}/rag16_by_article_{stamp}.csv", encoding="utf-8-sig")
lvl.to_csv(f"{OUT_DIR}/rag16_by_level_{stamp}.csv", encoding="utf-8-sig")
exp_res.to_csv(f"{OUT_DIR}/round1_extra_summary_{stamp}.csv", encoding="utf-8-sig")
detail = df[["문항ID", "평가 용도", "추출 그룹", "src", "et5", "틀린 말", "고친 말", "et5_status",
             "query", "query_rw", "query_enr", "gold"]].copy()
for name, n, qcol, ck in EXPS + [(f"#{c['#']}", c["#"], None, c["청킹"]) for c in COMBOS]:
    rk_all = EXP_RANKS[name] if name in EXP_RANKS else RANKS[n]
    labs = [c["label"] for c in CHUNKS[ck]]
    detail[f"{name} top3"] = [", ".join(labs[i] for i in rk[:TOPK_EVAL]) if rk else "" for rk in rk_all]
detail.to_csv(f"{OUT_DIR}/detail_{stamp}.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT_DIR)